In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("HW_2b_classes_and_simulation.ipynb")

# Homework 2b: Half car parameters, roads, and vibration

Slides for the homework: https://docs.google.com/presentation/d/1y6dZiGuSJBfha_5wPd5DEESuZWS8VUTdkc9zYLLOIoI/edit?usp=sharing

This is the second half of Homework 2. In part a you built a `HalfCar` in **`car.py`** that can draw and simulate a half car. Bring that same `car.py` into this folder (it should already be here if you are continuing in `Week_5_classes`).

You do **not** need to re-implement drawing, `B()`, or `simulate()`. This notebook provides a working `animate_car` and the Lab 6 bump road so you can jump into fixing inertia, plotting vibration, and comparing cars.

The two edits to `car.py` in this homework are marked with `# TODO HW 2b`; everything else happens in this notebook.


In [ ]:
# Imports
import copy
import numpy as np
from scipy.stats import gamma
from scipy.signal import StateSpace, lsim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
# Enable animations to work
%matplotlib widget

%load_ext autoreload
%autoreload 2
from car import QuarterCar, HalfCar


In [ ]:
# Lab 6 bump road (provided for HW 2b)
x_r_1 = [0, 5]
y_r_1 = [0, 0]
x_r_2 = [5.01, 5.5]
y_r_2 = [0, 0.3]
x_r_3 = [5.51, 6.0]
y_r_3 = [0.3, 0]
x_r_4 = [6.01, 11]
y_r_4 = [0, 0]
X_r = np.concatenate((x_r_1, x_r_2, x_r_3, x_r_4))
Y_r = np.concatenate((y_r_1, y_r_2, y_r_3, y_r_4))


In [ ]:
def animate_car(car, velocity, dt, x_r, y_r, playback_speed=1.0):
    """
    Animate the given half car going down the given road.
    @param car The `HalfCar` to simulate and draw.
    """
    car.simulate(velocity=velocity, dt=dt, x_r=x_r, y_r=y_r)
    fig, axs = plt.subplots()

    y_lim_max = np.max(y_r) + (car.front.y_s_static * 2)
    x_size_max = (car.a_f + car.a_b) * 2
    fig_width_and_height = max(y_lim_max, x_size_max)

    def draw_car_frame(i):
        axs.clear()
        axs.set_ylim([-0.1, fig_width_and_height])
        axs.set_xlim(np.array([car.x_r[i] - (fig_width_and_height / 2), car.x_r[i] + (fig_width_and_height / 2)]) + car.a_f)
        axs.plot(x_r, y_r, 'k')
        car.draw(axs=axs, x=car.x_r[i], y_rf=car.y_rf[i], y_uf=car.y_uf[i], y_rb=car.y_rb[i], y_ub=car.y_ub[i], y_s=car.y_s[i], a_f=car.a_f, a_b=car.a_b, theta=car.theta[i])

    return animation.FuncAnimation(fig, draw_car_frame, frames=len(car.x_r), interval=car.dt / playback_speed, repeat=False)


In [ ]:
# A default half car for drawings/animations that do not need custom physics
half_car = HalfCar()


## Part 3: Fixing the simulation.

The simulation hardcodes the mass moment of inertia as 1100 $kg.m^2$, but that isn't correct for every car, since the length and weight of the car influences this value.

Change `HalfCar.__init__` in **`car.py`** to approximate the mass moment of inertia of the car as a rod with an evenly distributed mass rotating about its end. Use the length of the car ($a_f + a_b$) as the length of the rod, and the total sprung mass (`m_s()`) as its mass.

Look for the `# TODO HW 2b` comment next to `self.Iy = 1100`.


In [ ]:
# GUIDES HW 2b: Follow the # TODO HW 2b comment in HalfCar.__init__ in car.py to replace the hardcoded
# Iy = 1100 with the rod approximation. Re-run this cell after editing car.py.

# 10m long stretch limo that weighs ~7000 lbs.
stretch_limo_model = HalfCar(
    front=QuarterCar(m_u=53, c_s=20000, k_s=200000, k_t=300000, m_s=1587),
    back=QuarterCar(m_u=76, c_s=20000, k_s=204000, k_t=300000, m_s=1587),
    a_f=4.0,
    a_b=6.0,
)

print(f"Iy: {stretch_limo_model.Iy}")


In [ ]:
import copy
# Copies the model and fixes Iy to the broken value so you can compare before/after a fix.
broken_model = copy.deepcopy(stretch_limo_model)
broken_model.Iy = 1100

# Animates this super long car on a longer version of X_r with a 3.8x larger peak.
# We are using a larger dt here because the animation is slow with the large plot size.
anim = animate_car(car=broken_model, velocity=10, dt=20, x_r=X_r * 5, y_r=Y_r * 3.8)
plt.show()


In [ ]:
# animate the car with your Iy fix
anim = animate_car(car=stretch_limo_model, velocity=10, dt=20, x_r=X_r * 5, y_r=Y_r * 3.8)
plt.show()


In [ ]:
grader.check("half_car_simulation_fix")

<!-- BEGIN QUESTION -->

GUIDES: With the mass moment of inertia calculation fixed, how does the animation above change? Write your answer in the panel below.


<!-- END QUESTION -->

## Part 4: Visualizing vibration

Oftentimes, the goal of simulation is to affordably test a design before building the real thing. For cars, one might use a simulation to try to minimize vibrations.

We can visualize the vibration of our half cars by plotting the delta between the optimal no-vibration y position of the sprung mass in the front and the back and the actual y position on a given road.

Recall how we determine the `y` position of the sprung mass in `HalfCar`'s `draw` function. For simplicity, let's focus on the back tire:

```python
y_sb = y_s + (a_b * theta) + self.back.y_s_static
```

Or, in fancy math notation:

$$y_{sb} = y_s + (a_b * \theta) + y_{s_{static}}$$

We want to know the delta between the car with _no vibration_ and the actual car.

No vibration would be:

$$y_{static} = y_{rb} + y_{s_{static}}$$

So, the delta would be:

$$y_{sb} - y_{static} = y_s + (a_b * \theta) + y_{s_{static}} - (y_{rb} + y_{s_{static}}) = y_s + (a_b * \theta) - y_{rb}$$

Your task is to perform this calculation on the Numpy arrays in the simulation output and graph it. Compare the vibrations of the broken limo from part 3 and the fixed limo.

In [ ]:
# Perform the simulation to get the output.
stretch_limo_model.simulate(velocity=10, dt=4, x_r=X_r * 5, y_r=Y_r * 3.8)
broken_model.simulate(velocity=10, dt=4, x_r=X_r * 5, y_r=Y_r * 3.8)


In [ ]:
# GUIDES: Write code to calculate and plot the vibration of the non-broken and broken limo. You should use two subplots -- one for each limo -- and fix the axes to be the same.
# The first plot should be the non-broken limo.
fig_vib, axs_vib = (..., ...)



In [ ]:
# GUIDES: Same plot as before, but use `sprung_mass_displacement`.


In [ ]:
grader.check("visualize_vibration")

<!-- BEGIN QUESTION -->

GUIDES: How does the vibration differ between the two outputs in the previous problem? Write your answer below

<!-- END QUESTION -->

## Part 5a: Cars

Now that we have a working half-car simulation, let's play with it a bit. Let's define two different half cars and ride them down three different roads -- two that we provide, and one that you will define. We can then plot their vibrations and compare.

The first half car will be a normal internal combustion engine, and the second will be an electric car.

For simplicity, we will be modeling the half cars with the weight of a full car.

### Internal combustion engine: 2025 Honda Civic

We will base our internal combustion engine model on a 2025 Honda Civic.

The Civic puts about 60% of its weight in the front, and 40% of its weight in the back, and its curb weight is about 1331kg ([source](https://www.dellahonda.net/how-much-does-a-honda-civic-weigh.html#:~:text=Honda%20Civic%20models%20have%20a,with%20good%20handling%20and%20balance.)).

Curb weight includes both sprung and unsprung masses. We will assume that the unsprung mass is about 90kg total, with 60% in the front, which seems in line with estimations from similar cars ([source](https://www.civicxi.com/forum/threads/2023-civic-si-spring-rates-motion-ratio-ride-frequencies-autocross.52362/)).

$$m_{curb} = m_s + m_u = 1331 - 90kg$$

$$m_u = 90kg$$

$$m_s = m_{curb} - m_u = 1331kg - 90kg = 1241kg$$

$$m_{uf} = 0.6 * m_u$$

$$m_{ub} = 0.4 * m_u$$

$$m_{sf} = 0.6 * m_s$$

$$m_{sb} = 0.4 * m_s$$


The 2025 Honda Civic is ~4.69m long. With 60% of the weight in the front, that places the center of mass closer to the front and further from the back:

$$l = 4.69m$$

$$a_f = 0.4 * l$$

$$a_b = 0.6 * l$$

For the suspension, we'll make a few educated guesses. Calculating the spring constant is:

$$k_s = \frac{4\pi^2f^2m_s}{mr^2}$$

...where $f$ is the natural frequency of the spring (in Hz), $m_s$ is the sprung mass, and $mr$ is the motion ratio of spring to wheel (how much the spring travels relative to the wheel).

The natural frequency of a passenger car is around 1Hz, and this isn't a super heavy car, so let's use $f=0.7 Hz$ for this car. And the motion ratio is frequently near 1, so let's arbitrarily decide $mr=0.8$ as the motion ratio.

That would make $k_s$:

$$k_{sf} = \frac{4\pi^2*0.7Hz^2*0.6 * 1241kg}{0.8^2} = 22506 N/m$$

$$k_{sb} = \frac{4\pi^2*0.7Hz^2*0.4 * 1241kg}{0.8^2} = 15004 N/m$$

To keep the car from vibrating excessively, we're going to add _damping_. For this we will need to set reasonable values for $c_{sf}$ and $c_{sb}$.

The damping constants are defined as:

$$c_s = 2 * m_s * dr * f * g$$

...where $f$ is the frequency of the spring _in radians_, $dr$ is the _damping ratio_, and $g$ is gravity (9.81 m/s).

The frequency of the suspension springs of the Civic is 0.7 Hz. We can convert Hz to radians by multiplying them by $2\pi$. However, we need the damping ratio.

The damping ratio of a regular car is often between 0.2 and 0.25 ([source](https://www.researchgate.net/figure/ehicle-typical-damping-coefficient_tbl1_245401813)), so we will arbitrarily use 0.25 for the Civic.

Thus, the damping constants are:

$$c_{sf} = 2 * m_{sf} * 0.25 * (0.7 Hz * 2\pi rads/Hz) * 9.81 m/s = 16063 Ns/m$$

$$c_{sb} = 2 * m_{sb} * 0.25 * (0.7 Hz * 2\pi rads/Hz) * 9.81 m/s = 10708.7 Ns/m$$


Finally, we'll use $k_{tf} = k_{tb} = 200000 N/m$, which is a reasonable baseline for a passenger car.

Now we can define the car!

In [ ]:
# GUIDES: Define the internal combustion engine car using the values above.
combustion_car_model = ...


In [ ]:
grader.check("cars_combustion")

### Electric car: 2025 Hyundai Ioniq 6

For our electric car, we'll choose another sedan: Hyundai's Ioniq 6. As an electric vehicle, this car is considerably heavier than the Civic due to battery weight -- a 1985 kg curb weight. We'll assume 120 kg of that curb weight is unsprung weight due to the heavier tires needed to support that weight.

Unlike the Civic, the Ioniq 6 is evenly balanced, with 50% of its weight falling on the front tires, and 50% on the back tires. That's because most of the weight in an EV is from batteries, which can be broken up to balance the vehicle's weight. This also makes our calculations simpler, since the front half of the car will use the same constants as the back half of the car!

Thus:

$$m_{sf} = m_{sb} = 0.5 * (1985 kg - 120 kg)$$

$$m_{uf} = m_{ub} = 0.5 * 120 kg$$

The Ioniq 6 is 4.85m long, making:

$$a_f = a_b = 0.5 * 4.85 m$$

For suspension, we'll once again use a motion ratio of 0.8, but up the frequency of the spring to 1.0 Hz, making:

$$k_{sf} = k_{sb} = \frac{4\pi^2*1.0Hz^2*0.5 * 1865 kg}{0.8^2} = 57521 N/m$$

Since this car is 50% heavier than the Civic, we'll increase the tire spring constant by 50%:

$$k_{tf} = k_{tb} = 300000 N/m$$

For the damping constant, we have a spring with frequency 1.0 Hz, and we will arbitrarily use 0.2 as its damping ratio, making:

$$c_{sf} = c_{sb} = 2 * m_{sf} * 0.2 * (1.0 Hz * 2\pi rads/Hz) * 9.81 m/s = 22991 Ns/m$$

Now you can define this car!

In [ ]:
# GUIDES: Define the electric car model using the values above.
electric_car_model = ...


In [ ]:
grader.check("cars_electric")

## Part 5b: Roads

We will provide two roads to you, and you will define a third road. Then, you will simulate driving the cars through the roads, and plot their vibrations.

In [ ]:
# Hilly road
X_r_test_1 = np.array([0, 7, 15, 20, 25, 40])
Y_r_test_1 = np.array([0, 0, 3, 2, 0, 0 ])

# Big spike
X_r_test_2 = np.array([0, 7, 12, 15, 20, 25, 40])
Y_r_test_2 = np.array([0, 0, 0, 3, 0, 0, 0 ])

# GUIDES: Define your own road! It should be at least 40 meters long.
X_r_test_3 = ...
Y_r_test_3 = ...

In [ ]:
# Check code: You can view animations of your car model here before proceeding to plot
test_anim = animate_car(car=electric_car_model, velocity=10, dt=10, x_r=X_r_test_3, y_r=Y_r_test_3)
plt.show()


In [ ]:
# GUIDES: Make a 3x1 plot showing the cars' vibrations going down each road. Each plot should be its own road,
# with two lines: one per car. Label each plot and each line.
# Simulate one road at a time, then plot, because simulate() stores results on the car.
fig_vib3, axs_vib3 = (..., ...)



In [ ]:
grader.check("cars_roads")

<!-- BEGIN QUESTION -->

## Part 6, extra credit: Designing cars with fewer vibrations

The cars we provided mimic actual cars out there, but are not perfect for reducing cabin vibration.

Can you design an even more stable car cabin for the test roads?

In [ ]:
# GUIDES: Define a car model for your car.
stable_car_model = ...



In [ ]:
# GUIDES: Graph the vibration of your car against the damped electric and combustion car models on the three test roads.


<!-- END QUESTION -->

## Hours and collaborators
Required for every assignment - fill out before you hand-in.

Listing names and websites helps you to document who you worked with and what internet help you received in the case of any plagiarism issues. You should list names of anyone (in class or not) who has substantially helped you with an assignment - or anyone you have *helped*. You do not need to list TAs.

Listing hours helps us track if the assignments are too long.

In [ ]:
import os

# List of names (creates a set)
worked_with_names = {"not filled out"}
# List of URLS FA26 (creates a set)
websites = {"not filled out"}
# Approximate number of hours, including lab/in-class time
hours = -1.5

# VS Code stores the path in a special global variable
if '__vsc_ipynb_file__' in globals():
    notebook_path = globals()['__vsc_ipynb_file__']
    notebook_name = os.path.basename(notebook_path)
    json_name = notebook_name[:-6] + "_source.json"
    if not os.path.exists(json_name):
        print(f"Could not find the json file {json_name}; make sure the required VSCode extension is installed and running")


In [ ]:
grader.check("hours_collaborators")

### To submit

Double check your plots.

- Submit this .ipynb file, the .json file, and **`car.py`** through Gradescope, to **HW 2b** (classes and simulation)

Failures: None expected
